In [ ]:
import sys, os, glob, shutil, time, json, gc
import numpy as np, pandas as pd
t0 = time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:6.0f}с] {m}", flush=True)
os.makedirs("/kaggle/working/src", exist_ok=True)
os.makedirs("/kaggle/working/models", exist_ok=True)
code = os.path.dirname(glob.glob("/kaggle/input/**/pair_features.py", recursive=True)[0])
for p in glob.glob(code + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(code + "/*.json"): shutil.copy(p, "/kaggle/working/models/")
# Версии из отправленного архива кладутся поверх: именно на них обучена структурная модель.
struct = os.path.dirname(glob.glob("/kaggle/input/**/pair_boost_hybrid.npz", recursive=True)[0])
for p in glob.glob(struct + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(struct + "/*.npz"): shutil.copy(p, "/kaggle/working/models/")
open("/kaggle/working/src/__init__.py", "a").close()
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")
from src.pair_features import build_matrix, feature_names
from src.measure_features import measures, compare_measures, MEASURE_FEATURES
from src.features import extract_model_features
from src.model import BoostedPairModel
from src.export_boost import export, save, predict_proba
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
from scipy.stats import rankdata

fold = os.path.dirname(glob.glob("/kaggle/input/**/llm_valid_pairs.parquet", recursive=True)[0])
sc = os.path.dirname(glob.glob("/kaggle/input/**/ce_relaxed.npy", recursive=True)[0])
pairs = pd.read_parquet(fold + "/llm_valid_pairs.parquet")
items = pd.read_parquet(fold + "/llm_valid_items.parquet")
y = (pairs["target"].to_numpy() > 0).astype(np.int8)
cat = pairs["id1"].map(dict(zip(items.id, items.category.astype(str)))).astype(str).to_numpy()
known = sorted(set(items.category.astype(str)))
log(f"пар {len(pairs):,}, доля+ {y.mean():.3f}")

names = list(feature_names(False))
X = np.zeros((len(pairs), len(names)), dtype=np.float32)
for c in known:
    rows = np.flatnonzero(cat == c)
    if not len(rows): continue
    sub = items[items.category.astype(str) == c].reset_index(drop=True)
    X[rows] = build_matrix(sub, pairs.iloc[rows].reset_index(drop=True), known, with_neighbours=False)
    del sub; gc.collect()
log("парные признаки готовы")

t = time.perf_counter()
legacy = extract_model_features(pairs[["id1", "id2"]], items)
primary = BoostedPairModel("models/pair_boost_hybrid.npz").predict_probability(legacy, cat)
aux = BoostedPairModel("models/pair_boost_hybrid_aux.npz").predict_probability(legacy, cat)
structural = (0.8 * primary + 0.2 * aux).astype(np.float32)
del legacy; gc.collect()
log(f"структурная модель за {time.perf_counter()-t:.0f}с")

M = {int(i): measures(a) for i, a in zip(items.id, items.attributes)}
MX = np.array([[r[n] for n in MEASURE_FEATURES] for r in
               (compare_measures(M[x], M[z]) for x, z in zip(pairs.id1, pairs.id2))], dtype=np.float32)
ALL = ["ce_relaxed", "ce_combo", "ce_spec", "ce_self", "ce_e5", "ce_balanced",
       "ce_ru2", "ce_best", "ce_lr3e5", "ce_ru"]
CE = {n: np.load(f"{sc}/{n}.npy").astype(np.float32) for n in ALL}
codes = np.array([known.index(c) if c in known else -1 for c in cat], dtype=np.float32)
masks = {c: cat == c for c in np.unique(cat)}
def rk(s):
    o = np.empty(len(s), np.float32)
    for m in masks.values(): o[m] = rankdata(s[m]) / m.sum()
    return o
def macro(s, rate=0.111, seeds=8):
    vals = []
    for seed in range(seeds):
        rng = np.random.default_rng(seed); per = []
        for m in masks.values():
            rows = np.flatnonzero(m); pos, neg = rows[y[rows]==1], rows[y[rows]==0]
            keep = min(len(pos), max(5, int(round(rate/(1-rate)*len(neg)))))
            ch = np.concatenate([rng.choice(pos, keep, replace=False), neg])
            per.append(average_precision_score(y[ch], s[ch]))
        vals.append(np.mean(per))
    return float(np.mean(vals)), float(np.std(vals))

PARAMS = dict(max_iter=500, max_leaf_nodes=63, learning_rate=0.06, l2_regularization=1.0,
              early_stopping=False, random_state=0)
half = np.random.default_rng(5).permutation(len(y)) % 2
def honest(subset):
    M = np.column_stack([X, MX] + [CE[e] for e in subset] + [structural, codes]).astype(np.float32)
    p = np.zeros(len(y))
    for h in (0, 1):
        tr, te = half != h, half == h
        p[te] = HistGradientBoostingClassifier(**PARAMS).fit(M[tr], y[tr]).predict_proba(M[te])[:, 1]
    return macro(rk(p))

SETS = {
    "4 (нынешние)": ALL[:4],
    "6": ALL[:6],
    "8": ALL[:8],
    "10 (все)": ALL,
}
results = {}
for tag, subset in SETS.items():
    t = time.perf_counter(); mu, sd = honest(subset); results[tag] = (mu, sd, subset)
    log(f"  {tag:<14} {mu:.6f} ± {sd:.6f}   ({len(subset)} энкодеров, {time.perf_counter()-t:.0f}с)")

# Жадный отбор: добавляем по одному того, кто даёт больше всего, пока растёт.
chosen = list(ALL[:4]); best = results["4 (нынешние)"][0]
rest = [e for e in ALL if e not in chosen]
while rest:
    gains = []
    for e in rest:
        mu, _ = honest(chosen + [e]); gains.append((mu, e))
    mu, e = max(gains)
    if mu <= best + 0.001:
        log(f"  жадный отбор остановлен: лучший кандидат {e} даёт {mu:.6f} против {best:.6f}")
        break
    chosen.append(e); rest.remove(e); best = mu
    log(f"  + {e}: {mu:.6f}  (всего {len(chosen)})")
log(f"\nитог: {len(chosen)} энкодеров {chosen} -> {best:.6f}")
json.dump({"chosen": chosen, "score": best,
           "table": {k: v[0] for k, v in results.items()}},
          open("/kaggle/working/encoder_choice.json", "w"), ensure_ascii=False, indent=1)
